# 15 · Dynamic Shopping Cart Recommendation Demo

Run All in VS Code. Matches the professor's example: start with an empty cart, add Bosch D7050, then add DSC PG9914, and recompute suggestions after **every** change. Prints ranked products and transparent metrics. Also tests unknown/unseen combinations.

**All product associations are from a clearly labeled synthetic transaction fixture.** They do not establish real product interoperability.

In [1]:
%run 14_market_basket_engine.ipynb

Database initialized: /Users/nehlinshanila/Downloads/group2-course-rag-engine/.runtime/group2.sqlite3
Apriori basket recommender ready; call seed_market_basket_demo() to load synthetic transactions.


In [2]:
from IPython.display import display, Markdown
from pprint import pprint
seed_result=seed_market_basket_demo(PROJECT_ID,replace=True)
print('SYNTHETIC SOURCE LOADED:');pprint(seed_result)
print('Catalog:',len(basket_inventory(PROJECT_ID)['products']),'products')

SYNTHETIC SOURCE LOADED:
{'is_synthetic': True,
 'line_count': 575,
 'product_count': 12,
 'project_id': '77777777-7777-4777-8777-777777777777',
 'source_type': 'SYNTHETIC_FIXTURE',
 'transaction_count': 248}
Catalog: 12 products


In [3]:
def show_cart_step(items):
    result=market_basket_recommend(PROJECT_ID,items,top_k=5,min_support=.05,min_confidence=.4,min_lift=1.05)
    display(Markdown('### Cart: '+(' + '.join(items) if items else '(empty)')))
    if result['abstained']:
        print('No recommendations:',result['reason'])
    else:
        for i,rec in enumerate(result['recommendations'],1):
            print(f"{i}. {rec['product_name']} — score {rec['score']:.3f}; confidence={rec['confidence']:.2f}; lift={rec['lift']:.2f}; support={rec['support']:.2f}")
            print('   Evidence:',rec['evidence'],' | ',rec['reason'])
    return result

empty=show_cart_step([])
one=show_cart_step(['BOSCH-D7050'])
two=show_cart_step(['BOSCH-D7050','DSC-PG9914'])
unknown=show_cart_step(['BOSCH-D7050','AXIS-M3046'])
assert empty['abstained'] and unknown['abstained']
assert not one['abstained'] and not two['abstained']
assert one['recommendations'] != two['recommendations']
print('\nPASS: Recomputation and safe abstention for unseen cart combinations.')

### Cart: (empty)

No recommendations: empty_cart


### Cart: BOSCH-D7050

1. DSC PG9914 Motion Detector — score 0.731; confidence=0.84; lift=2.13; support=0.32
   Evidence: {'antecedent_transactions': 95, 'joint_transactions': 80, 'total_transactions': 248}  |  Co-purchased in 80 of 95 historical orders containing the full current cart.
2. Bosch B810 Wireless Receiver — score 0.438; confidence=0.44; lift=1.77; support=0.17
   Evidence: {'antecedent_transactions': 95, 'joint_transactions': 42, 'total_transactions': 248}  |  Co-purchased in 42 of 95 historical orders containing the full current cart.


### Cart: BOSCH-D7050 + DSC-PG9914

1. Bosch B810 Wireless Receiver — score 0.515; confidence=0.53; lift=2.10; support=0.17
   Evidence: {'antecedent_transactions': 80, 'joint_transactions': 42, 'total_transactions': 248}  |  Co-purchased in 42 of 80 historical orders containing the full current cart.


### Cart: BOSCH-D7050 + AXIS-M3046

No recommendations: no_historical_cart_pattern

PASS: Recomputation and safe abstention for unseen cart combinations.


In [4]:
# Optional interactive cart inside VS Code Jupyter (requires ipywidgets):
try:
    import ipywidgets as widgets
    catalog=basket_inventory(PROJECT_ID)['products']
    box=widgets.SelectMultiple(options=[(p['name'],p['sku']) for p in catalog],rows=10,
                               description='My cart:',layout=widgets.Layout(width='550px'))
    out=widgets.Output()
    def on_cart_change(change):
        with out:
            out.clear_output()
            result=market_basket_recommend(PROJECT_ID,list(change['new']))
            if result['abstained']:
                print('No recommendation:',result['reason'])
            else:
                for item in result['recommendations']:
                    print(f"{item['product_name']} | confidence {item['confidence']:.1%} | lift {item['lift']:.2f}")
    box.observe(on_cart_change,names='value')
    display(widgets.VBox([widgets.HTML('<b>Try selecting items to update the basket</b>'),box,out]))
except ImportError:
    print('Optional interactive widget unavailable; demo above runs without it. Install ipywidgets to enable.')